In [1]:
""" M5 Forecasting: Full Exploratory Data Analysis """

# 1. SETUP & IMPORTS
# ------------------------------------------
# Add the project root to the system path so we can import from 'src'
# Assumes this script is running in backend/notebooks/ or similar
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import paths

# Configure plotting aesthetics for high-quality outputs
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Define and create the directory for saving plots
PLOTS_DIR = paths.REPORTS_DIR / "plots"\nPLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Setup Complete. Plots will be saved to: {PLOTS_DIR}")

✅ Setup Complete. Plots will be saved to: /Users/hakan/Coding/ie-capstone-m5/backend/experiments/plots


In [2]:
# 2. LOAD PROCESSED DATA
# ------------------------------------------
# We use Polars 'scan_parquet' for lazy loading. This is incredibly fast
# because it doesn't load the whole file into RAM until we call .collect().
print("\n⏳ Loading Data...")
sales_path = paths.PROCESSED_DATA_DIR / "melted_sales.parquet"
calendar_path = paths.PROCESSED_DATA_DIR / "calendar.parquet"
prices_path = paths.PROCESSED_DATA_DIR / "sell_prices.parquet"

q_sales = pl.scan_parquet(sales_path)
q_cal = pl.scan_parquet(calendar_path)

print("   Data Schema:")
print(q_sales.schema)


⏳ Loading Data...
   Data Schema:
Schema([('id', String), ('item_id', Categorical), ('dept_id', Categorical), ('cat_id', Categorical), ('store_id', Categorical), ('state_id', Categorical), ('d', String), ('sales', Int16)])


/var/folders/mr/lg1tn35x6c362w3gh039ljn00000gn/T/ipykernel_66927/2264646841.py:14: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(q_sales.schema)


In [3]:
# 3. VISUALIZATION 1: GLOBAL SALES TREND
# ------------------------------------------
# Goal: Aggregating 50M+ rows to see the total sales volume over 5 years.
print("\n📊 Generatng Global Sales Trend...")

global_daily_sales = (
    q_sales
    # Join with calendar to get actual 'date' objects instead of just 'd_1'
    .join(q_cal, left_on="d", right_on="d")
    # Group by date and sum up all sales
    .group_by("date")
    .agg(pl.col("sales").sum().alias("total_sales"))
    .sort("date")
    .collect() # Execute the query
)

# Plotting
plt.figure(figsize=(15, 5))
plt.plot(global_daily_sales["date"], global_daily_sales["total_sales"], alpha=0.7, label="Daily Sales")

# Calculate a 28-day rolling average to smooth out the weekly noise
rolling = global_daily_sales["total_sales"].rolling_mean(window_size=28)
plt.plot(global_daily_sales["date"], rolling, color='red', linewidth=2, label="28-Day Moving Avg")

plt.title("Global Daily Sales Trend (All Stores)")
plt.legend()

# Save
save_path = PLOTS_DIR / "global_sales_trend.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"   Saved: {save_path}")
plt.close() # Close plot to free memory


📊 Generatng Global Sales Trend...
   Saved: /Users/hakan/Coding/ie-capstone-m5/backend/experiments/plots/global_sales_trend.png


In [4]:
# 4. VISUALIZATION 2: SALES BY STORE
# ------------------------------------------
# Goal: Compare performance across different stores (CA, TX, WI).
print("\n📊 Generating Sales by Store...")

store_sales = (
    q_sales
    .join(q_cal, left_on="d", right_on="d")
    .group_by(["date", "store_id"])
    .agg(pl.col("sales").sum().alias("total_sales"))
    .sort("date")
    .collect()
)

plt.figure(figsize=(15, 6))
# We convert to Pandas here because Seaborn handles categorical hue/legend better
sns.lineplot(data=store_sales.to_pandas(), x="date", y="total_sales", hue="store_id", alpha=0.5)
plt.title("Sales Volume by Store")

# Save
save_path = PLOTS_DIR / "sales_by_store.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"   Saved: {save_path}")
plt.close()


📊 Generating Sales by Store...
   Saved: /Users/hakan/Coding/ie-capstone-m5/backend/experiments/plots/sales_by_store.png


In [5]:
# 5. ANALYSIS: DATA SPARSITY
# ------------------------------------------
# Goal: Check how many rows have 0 sales. High sparsity influences our model choice.
print("\n🧮 Calculating Sparsity...")

zero_count = (
    q_sales
    .select([
        (pl.col("sales") == 0).sum().alias("zero_sales"),
        pl.count("sales").alias("total_rows")
    ])
    .collect()
)

pct_zeros = (zero_count["zero_sales"][0] / zero_count["total_rows"][0]) * 100
print(f"   Percentage of rows with ZERO sales: {pct_zeros:.2f}%")
# Insight: If >50%, standard MSE loss might be unstable.


# 6. VISUALIZATION 3: SINGLE ITEM DRILL-DOWN
# ------------------------------------------
# Goal: Visualize the raw, noisy signal of a single product to see what we are predicting.
sample_item = "FOODS_3_090"
sample_store = "CA_1"
print(f"\n📊 Generating Drill-down for {sample_item} at {sample_store}...")

single_item_history = (
    q_sales
    .filter(
        (pl.col("item_id") == sample_item) & 
        (pl.col("store_id") == sample_store)
    )
    .join(q_cal, left_on="d", right_on="d")
    .select(["date", "sales"])
    .sort("date")
    .collect()
)

plt.figure(figsize=(15, 4))
plt.plot(single_item_history["date"], single_item_history["sales"], marker='.', linestyle='none', alpha=0.5)
plt.title(f"Daily Sales for {sample_item} at {sample_store}")
plt.ylabel("Units Sold")

# Save
save_path = PLOTS_DIR / f"single_item_sales_{sample_item}.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"   Saved: {save_path}")
plt.close()

print("\n✅ EDA Complete!")


🧮 Calculating Sparsity...
   Percentage of rows with ZERO sales: 68.20%

📊 Generating Drill-down for FOODS_3_090 at CA_1...
   Saved: /Users/hakan/Coding/ie-capstone-m5/backend/experiments/plots/single_item_sales_FOODS_3_090.png

✅ EDA Complete!
